In [2]:
from shopping_assistant.env import load_env
load_env()

True

In [50]:
from openai import OpenAI
from agents import function_tool, Agent, Runner

openai_client = OpenAI()

In [99]:
from typing import Annotated
from collections import Counter

class Cart:
    def __init__(self, user_id: str):
        self.user_id = user_id
        self.items = Counter()

    def add_item(self, item, quantity):
        self.items[item] += quantity
        return self

    def remove_item(self, item, quantity):
        cur_qty = self.items.get(item, 0)
        if cur_qty < quantity:
            raise ValueError("Not enough quantity in cart")
        if cur_qty == quantity:
            return self.empty_item(item)
    
        self.items[item] -= quantity
        return self

    def empty_cart(self):
        self.items.clear()
        return self

    def empty_item(self, item):
        self.items.pop(item, None)
        return self

    def view_cart(self):
        return dict(self.items)


def get_cart_action_tools_for_user(cart: Cart) -> list:

    @function_tool
    def add_to_cart(
        product_id: Annotated[str, "product_id"],
        quantity: Annotated[int, "quantity"] = 1
    ) -> dict:
        """Add a product to user's cart"""
        try:
            cart.add_item(product_id, quantity)
        except Exception as e:
            return "Error updating cart: " + str(e)
        
        return "Cart updated successfully. Current "

    @function_tool
    def remove_from_cart(product_id: str, quantity: Annotated[int, "quantity"]) -> dict:
        """Remove a product from user's cart"""
        return cart.remove_item(product_id, quantity)

    @function_tool
    def view_cart() -> list:
        """View user's cart"""
        return cart.view_cart()

    return [
        add_to_cart,
        remove_from_cart,
        view_cart
    ]




In [98]:
cart = Cart('123')

print(cart.view_cart())

cart.add_item('item_1', quantity=1)
print(cart.view_cart())

cart.remove_item('item_1', quantity=1)
print(cart.view_cart())



{}
{'item_1': 1}
{}


In [110]:
cart = Cart(user_id="123")

SHOPPING_AGENT_ACTIONS = {
    "cart": {
        "summary": "Manage your shopping cart (add or remove items, view cart)",
        "tools": get_cart_action_tools_for_user(cart),
    }
}

ACTION_SUMMARY = '\n'.join([
    f"{idx + 1}. Action=`{action_type}`: {info['summary']}"
    for idx, (action_type, info) in enumerate(SHOPPING_AGENT_ACTIONS.items())
])

ACTION_DETAILED = ''
for idx, (action_type, info) in enumerate(SHOPPING_AGENT_ACTIONS.items()):
    ACTION_DETAILED += f"{idx + 1}. Action=`{action_type}`: {info['summary']}:\n"
    for tool in info['tools']:
        ACTION_DETAILED += f"- `{tool.name}`: {tool.description}\n"

ACTION_TYPES = list(SHOPPING_AGENT_ACTIONS.keys())


SHOPPING_ACTION_TOOLS = [
    tool
    for action in SHOPPING_AGENT_ACTIONS.values()
    for tool in action['tools']
]

In [111]:
print(ACTION_DETAILED)

1. Action=`cart`: Manage your shopping cart (add or remove items, view cart):
- `add_to_cart`: Add a product to user's cart
- `remove_from_cart`: Remove a product from user's cart
- `view_cart`: View user's cart



In [112]:
SYSTEM_PROMPT = f"""
You are a helpful shopping assistant that can help users with their shopping needs.
Based on the user's query, you do the following:

Step 1. Parse the user's query to understand the user's intented action type out of the following list: {ACTION_TYPES})
Step 2. If no action type is identified, you respond verbatim with:

```
I'm sorry, I don't understand your query. I can help with the following actions:
{ACTION_SUMMARY}
```
Step 3. Based on the action type, you will call the appropriate action function from the following list:

{ACTION_DETAILED}

Step 4. After the action is completed, you will inform the user of the action result.

"""

In [116]:
l = get_cart_action_tools_for_user(Cart('123'))

In [123]:
Cart('123').view_cart()

{}

In [124]:
shopping_action_agent = Agent(
    name="shopping-action-agent",
    instructions=SYSTEM_PROMPT,
    tools=get_cart_action_tools_for_user(Cart('123')),
    model="gpt-4o-mini",
)

response = await Runner.run(shopping_action_agent, input="Can I know what's in my cart?")

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Your authentication token is not from a valid issuer.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_issuer"
  }
}


In [125]:
print(response.final_output)

Your cart is currently empty. If you’d like to add items, just let me know!


[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Your authentication token is not from a valid issuer.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_issuer"
  }
}


In [126]:
response = await Runner.run(shopping_action_agent, input="Can you add this 2 of this product to my cart? (item_524)")
print(response.final_output)

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Your authentication token is not from a valid issuer.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_issuer"
  }
}


I've added 2 of the product (item_524) to your cart successfully! If you need further assistance, just let me know.


[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Your authentication token is not from a valid issuer.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_issuer"
  }
}


In [106]:
response.raw_responses[0].output

[ResponseFunctionToolCall(arguments='{"product_id":"item_524","quantity":2}', call_id='call_NjtSNPgUHmsQfTldoeyhTHA1', name='add_to_cart', type='function_call', id='fc_04ce4cbc1f402a1c0169245c67a8fc819ead18dd6d6d6cd949', status='completed')]

In [127]:
response = await Runner.run(shopping_action_agent, input="What's in my cart")
print(response.final_output)

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Your authentication token is not from a valid issuer.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_issuer"
  }
}


You have 2 items of product ID **item_524** in your cart. If you need further assistance, feel free to ask!


[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Your authentication token is not from a valid issuer.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_issuer"
  }
}


In [128]:
response.raw_responses[0].output

[ResponseFunctionToolCall(arguments='{}', call_id='call_cdu4QKhfNdrc67agH3TDTVi1', name='view_cart', type='function_call', id='fc_09b6d466dfcb65540169245dd169ec819eb8b7b1cd994644db', status='completed')]

In [129]:
response = await Runner.run(shopping_action_agent, input="Can you remove item_161 from my cart and tell me what's left?")
print(response.final_output)

[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Your authentication token is not from a valid issuer.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_issuer"
  }
}
[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Your authentication token is not from a valid issuer.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_issuer"
  }
}


It looks like I couldn't remove item_161 from your cart because there wasn't enough quantity. However, your cart currently contains:

- item_524: 2 units.

If you need assistance with anything else, feel free to ask!


[non-fatal] Tracing client error 401: {
  "error": {
    "message": "Your authentication token is not from a valid issuer.",
    "type": "invalid_request_error",
    "param": null,
    "code": "invalid_issuer"
  }
}


In [133]:
response.raw_responses[0].output

[ResponseFunctionToolCall(arguments='{"product_id":"item_161","quantity":1}', call_id='call_haShMENr1OHAcMUethdvVLaH', name='remove_from_cart', type='function_call', id='fc_07f9e689ce283c3e0169245df90904819cbf0a6df1a78d1f5d', status='completed'),
 ResponseFunctionToolCall(arguments='{}', call_id='call_yYbOo67Suz47Mwj9TA8xCbO7', name='view_cart', type='function_call', id='fc_07f9e689ce283c3e0169245df96a14819c9d03a2b64b86fd47', status='completed')]